# DDPM을 밑바닥부터 구현하기 (MNIST, 28x28 흑백)

Ho et al., "Denoising Diffusion Probabilistic Models" (NeurIPS 2020)

구성
  1) NoiseSchedule : beta_t, alpha_t, alpha_bar_t 계산 + forward process q(x_t | x_0)
  2) TinyUNet      : 노이즈 eps_theta(x_t, t)를 예측하는 작은 U-Net
  3) train()       : Algorithm 1 (학습)
  4) sample()      : Algorithm 2 (샘플링)

In [ ]:
import math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

GPUS = tf.config.list_physical_devices("GPU")
print(f"device: {'GPU' if GPUS else 'CPU'}")

### 1) 노이즈 스케줄과 forward process

In [ ]:
class NoiseSchedule:
    """beta_t (linear schedule)와 그로부터 유도되는 값들을 미리 계산해 둡니다."""

    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        self.T = T
        self.betas = tf.linspace(beta_start, beta_end, T)              # beta_t
        self.betas = tf.cast(self.betas, tf.float32)
        self.alphas = 1.0 - self.betas                                 # alpha_t
        self.alpha_bars = tf.math.cumprod(self.alphas, axis=0)         # alpha_bar_t = prod alpha_s

    def q_sample(self, x0, t, eps):
        """q(x_t | x_0) 에서 한 번에 샘플링:
           x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
        """
        ab = tf.gather(self.alpha_bars, t)
        ab = tf.reshape(ab, [-1, 1, 1, 1])          # (B,) -> (B,1,1,1) 브로드캐스트
        return tf.sqrt(ab) * x0 + tf.sqrt(1 - ab) * eps

### 2) 노이즈 예측 네트워크 (작은 U-Net)

In [ ]:
def timestep_embedding(t, dim):
    """Transformer의 positional encoding과 같은 sinusoidal embedding. t: (B,) -> (B, dim)"""
    half = dim // 2
    freqs = tf.exp(-math.log(10000) * tf.range(half, dtype=tf.float32) / half)
    args = tf.cast(t, tf.float32)[:, None] * freqs[None, :]
    return tf.concat([tf.sin(args), tf.cos(args)], axis=1)


class ResBlock(layers.Layer):
    """Conv 두 번 + timestep embedding을 더해주는 residual block"""

    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.conv1 = layers.Conv2D(out_ch, 3, padding="same")
        self.conv2 = layers.Conv2D(out_ch, 3, padding="same")
        self.t_proj = layers.Dense(out_ch)                # t embedding -> 채널별 bias
        self.norm1 = layers.GroupNormalization(groups=8, axis=-1)
        self.norm2 = layers.GroupNormalization(groups=8, axis=-1)
        self.skip = layers.Conv2D(out_ch, 1) if in_ch != out_ch else None

    def call(self, x, t_emb):
        h = tf.nn.silu(self.norm1(self.conv1(x)))
        h = h + self.t_proj(t_emb)[:, None, None, :]     # 시간 정보 주입
        h = tf.nn.silu(self.norm2(self.conv2(h)))
        skip = self.skip(x) if self.skip is not None else x
        return h + skip


class TinyUNet(tf.keras.Model):
    """28x28 -> 14x14 -> 7x7 -> 14x14 -> 28x28 의 3단계 U-Net"""

    def __init__(self, ch=64, t_dim=128):
        super().__init__()
        self.t_dim = t_dim
        self.t_mlp = tf.keras.Sequential([
            layers.Dense(t_dim), layers.Activation("swish"), layers.Dense(t_dim),
        ])

        # 인코더 (다운샘플링)
        self.inc = layers.Conv2D(ch, 3, padding="same")
        self.down1 = ResBlock(ch, ch * 2, t_dim)      # 28x28
        self.down2 = ResBlock(ch * 2, ch * 4, t_dim)  # 14x14
        self.mid = ResBlock(ch * 4, ch * 4, t_dim)    # 7x7
        self.pool = layers.AveragePooling2D(2)
        # 디코더 (업샘플링) - skip connection 때문에 입력 채널이 2배
        self.up1 = ResBlock(ch * 4 + ch * 4, ch * 2, t_dim)
        self.up2 = ResBlock(ch * 2 + ch * 2, ch, t_dim)
        self.upsample = layers.UpSampling2D(2, interpolation="nearest")
        self.outc = layers.Conv2D(1, 1)

    def call(self, x, t, training=None):
        t_emb = self.t_mlp(timestep_embedding(t, self.t_dim))

        h0 = self.inc(x)                          # (B, 28, 28, ch)
        h1 = self.down1(h0, t_emb)                # (B, 28, 28, 2ch)
        h2 = self.down2(self.pool(h1), t_emb)     # (B, 14, 14, 4ch)
        h3 = self.mid(self.pool(h2), t_emb)       # (B, 7, 7, 4ch)

        u = self.upsample(h3)                     # 7 -> 14
        u = self.up1(tf.concat([u, h2], axis=-1), t_emb)
        u = self.upsample(u)                       # 14 -> 28
        u = self.up2(tf.concat([u, h1], axis=-1), t_emb)
        return self.outc(u)                        # 예측된 노이즈 eps_theta (B,28,28,1)


### 3) 학습

In [ ]:
def train(model, schedule, dataset, epochs=5, lr=2e-4):
    opt = tf.keras.optimizers.Adam(lr)

    @tf.function
    def train_step(x0):
        t = tf.random.uniform((tf.shape(x0)[0],), 0, schedule.T, dtype=tf.int32)  # t ~ Uniform
        eps = tf.random.normal(tf.shape(x0))                                       # eps ~ N(0, I)
        x_t = schedule.q_sample(x0, t, eps)                                        # forward process
        with tf.GradientTape() as tape:
            eps_hat = model(x_t, t)
            loss = tf.reduce_mean(tf.square(eps - eps_hat))                        # L_simple = ||eps - eps_theta||^2
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for epoch in range(epochs):
        for step, x0 in enumerate(dataset):
            loss = train_step(x0)
            if step % 100 == 0:
                print(f"epoch {epoch}  step {step:4d}  loss {loss.numpy():.4f}")



In [ ]:
(x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0            # [0,1]
x_train = x_train[..., None] * 2 - 1                   # [0,1] -> [-1,1], (N,28,28,1)

dataset = (tf.data.Dataset.from_tensor_slices(x_train)
            .shuffle(10000)
            .batch(128)
            .prefetch(tf.data.AUTOTUNE))

schedule = NoiseSchedule(T=1000)
model = TinyUNet()
model(tf.zeros((1, 28, 28, 1)), tf.zeros((1,), dtype=tf.int32))  # 더미 입력으로 가중치 생성
print(f"parameters: {sum(np.prod(v.shape) for v in model.trainable_variables) / 1e6:.2f}M")

train(model, schedule, dataset, epochs=5)

### 4) 샘플링 테스트

In [ ]:
def sample(model, schedule, n=64):
    x = tf.random.normal((n, 28, 28, 1))                    # x_T ~ N(0, I)
    for t in reversed(range(schedule.T)):                    # t = T-1, ..., 0
        t_batch = tf.fill((n,), t)
        eps_hat = model(x, t_batch, training=False)

        alpha, alpha_bar, beta = schedule.alphas[t], schedule.alpha_bars[t], schedule.betas[t]
        # 평균: mu = 1/sqrt(alpha_t) * (x_t - beta_t / sqrt(1 - alpha_bar_t) * eps_hat)
        mean = (x - beta / tf.sqrt(1 - alpha_bar) * eps_hat) / tf.sqrt(alpha)
        noise = tf.random.normal(tf.shape(x)) if t > 0 else tf.zeros_like(x)   # 마지막 step은 노이즈 없음
        x = mean + tf.sqrt(beta) * noise                     # sigma_t^2 = beta_t 선택
    return (tf.clip_by_value(x, -1, 1) + 1) / 2               # [-1,1] -> [0,1]


def save_image_grid(imgs, path, nrow=8):
    """(N, H, W, 1) 텐서를 nrow x ncol 격자 이미지로 저장합니다 (torchvision.save_image 대체)."""
    imgs = (imgs.numpy() * 255).astype(np.uint8)[..., 0]     # (N, H, W)
    n, h, w = imgs.shape
    ncol = math.ceil(n / nrow)
    grid = np.zeros((ncol * h, nrow * w), dtype=np.uint8)
    for i, img in enumerate(imgs):
        r, c = divmod(i, nrow)
        grid[r * h:(r + 1) * h, c * w:(c + 1) * w] = img
    tf.keras.utils.save_img(path, grid[..., None])

In [ ]:
imgs = sample(model, schedule, n=64)
save_image_grid(imgs, "ddpm_samples.png", nrow=8)

In [ ]:
from google.colab import files
files.download('ddpm_samples.png')